In [1]:
from crewai import Agent, LLM, Task, Crew
from crewai_tools import FileReadTool

from IPython.display import Markdown

/home/luukvanroessel/miniconda3/envs/ml-luuk/lib/python3.12/site-packages/pydantic/_internal/_config.py:295: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)
/home/luukvanroessel/miniconda3/envs/ml-luuk/lib/python3.12/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "schema" in "DatabricksQueryToolSchema" shadows an attribute in parent "BaseModel"
  warnings.warn(
/home/luukvanroessel/miniconda3/envs/ml-luuk/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: <built-in function callable> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error 

In [2]:
llm = LLM(model="ollama/llama3.2:1b",
          base_url="http://localhost:11434")

file_read_tool = FileReadTool()

In [3]:
news_website_scraper = Agent(
    role="Expert Stock News Scraper",
    goal=(
        "Scrape relevant news articles that could affect stock prices "
        "and feed them to the Feature Extractor for further processing."
    ),
    backstory=(
        "You are a pro at finding the most important and timely news articles "
        "that impact stock prices. You continuously scan multiple news outlets, "
        "social media platforms, and press releases to provide the most "
        "up-to-date information to the Feature Extractor for analysis."
    ),
    llm=llm,
    verbose=True,
)

feature_extractor = Agent(
    role="Expert News Article Feature Extractor",
    goal=(
        "Extract key features from the scraped news articles, including "
        "the relevant company(ies), a sentiment score, and any other "
        "potentially useful metadata (e.g., publication date, article length, "
        "keywords)."
    ),
    backstory=(
        "You are an NLP (Natural Language Processing) specialist with years of "
        "experience in analyzing text data. You have been hired by a hedge fund "
        "to take raw news articles and systematically transform them into "
        "structured data—ready for further analysis by other agents."
    ),
    llm=llm,
    verbose=True,
)

historian = Agent(
    role="Expert in Stock Market History",
    goal=(
        "Provide historical context for the features extracted from the news articles. "
        "Use your knowledge of past market patterns, past company performance, and "
        "macro events to add depth to the Feature Extractor’s structured output."
    ),
    backstory=(
        "You are an authority on stock market history. When provided with structured data "
        "from the Feature Extractor, you use your deep knowledge of past events to "
        "draw parallels, compare time periods, and suggest relevant context that "
        "could enrich the analysis."
    ),
    llm=llm,
    tools=[file_read_tool],
    verbose=True,
)

company_expert = Agent(
    role="Expert in Company Fundamentals",
    goal=(
        "Provide detailed insights into the fundamentals of the companies mentioned "
        "in the news articles. This includes financial data, key performance indicators, "
        "and any other relevant information that could help predict stock price movements."
    ),
    backstory=(
        "You are a financial analyst with a keen eye for company fundamentals. "
        "You dive deep into financial reports, earnings calls, and industry trends "
        "to provide a comprehensive view of the companies mentioned in the news articles. "
        "Your insights are crucial for understanding the underlying value of the stocks."
    ),
    llm=llm,
    tools=[file_read_tool],
    verbose=True,
)

stock_analyst = Agent(
    role="Senior Stock Market Analyst",
    goal=(
        "Use the extracted features from the Feature Extractor, combined with the "
        "historical insights from the Historian, to predict potential stock price "
        "movements and offer actionable insights."
    ),
    backstory=(
        "You are a seasoned equity research analyst, adept at synthesizing diverse data "
        "inputs, from sentiment scores and structured article features to broader "
        "market history. Armed with the Feature Extractor’s outputs and the Historian’s "
        "context, you transform raw information into trading strategies."
    ),
    llm=llm,
    verbose=True,
)


### Tasks

In [4]:
feature_extract_task = Task(
    description="""Extract key features from the scraped news article: {article}, including the 
        relevant company(ies), a sentiment score, and any other potentially useful metadata 
        (e.g., publication date, article length, keywords).""",
    agent=feature_extractor,
    expected_output="""Structured data with key features extracted from the news articles, 
        including company names, sentiment scores, and other relevant metadata.""",
)

historical_context_task = Task(
    description="""Provide historical context for the features extracted from {file_path}. 
        Use your knowledge of past market patterns, past company performance, and macro events 
        to add depth to the Feature Extractor’s structured output.""",
    agent=historian,
    tools=[file_read_tool],
    expected_output="""A 20 word description of historical insights and context that enrich the structured data 
        from the Feature Extractor, drawing parallels, comparing time periods, and suggesting 
        relevant historical context.""",
)

company_fundamentals_task = Task(
    description="""Provide detailed insights into the fundamentals of the companies mentioned in the 
        {file_path}. This includes financial data, key performance indicators, and any other 
        relevant information that could help predict stock price movements.""",
    agent=company_expert,
    tools=[file_read_tool],
    expected_output="""Comprehensive insights into the fundamentals of the companies mentioned in the news articles, 
        including financial data, key performance indicators, and other relevant information.""",
)

### Crew

In [9]:
article = """Tesla's China rival BYD's sales surged 29% last year, topping $100B

BYD (1211.HK) capped off a strong 2024 with positive results as Tesla's main rival in China sets up for another big year.

BYD's full year jumped 29% to 777 billion yuan ($107 billion) vs. 776 billion yuan estimated, crossing $100 billion in sales for the first time. Net income rose 34% to 40.3 billion yuan ($5.56 billion), topping the 39.5 billion yuan ($5.5 billion) estimated, per Bloomberg consensus.

BYD stock trading in Hong Kong closed up 3% on Monday, with shares up a whopping 50% year to date.

BYD's success in 2024 comes as Tesla, the EV leader, saw sales drop for the first time. BYD's total deliveries hit 4.27 million compared to Tesla's 1.79 million, though this is a mix of hybrid and battery electric vehicles (BEV). BYD sold 1.76 million BEVs globally last year.

In China, BYD's home market and a crucially important one for Tesla, BYD sales continually topped Tesla and stretched that lead over the year and into 2025.

BYD's prowess in building cheaper cars — its Seal EV costs around $12,000 — and advancing technology like its "God's Eye" self-driving software has made the company's cars a compelling option in China's highly competitive auto market."

Over the weekend, BYD launched a new sedan, dubbed the Qin L, that would take on Tesla's Model 3. The Qin L features the God's Eye self-driving feature, has a range of 545 kilometers (approximately 340 miles), and would start at 119,800 yuan ($16,524), or half the cost of the Model 3.

Last week, BYD also said a new battery and charging system — dubbed the Super e-Platform — can charge at peak speeds of 1,000kW, providing around 250 miles of range in just five minutes, per BYD chair and founder Wang Chuanfu.

By contrast, Tesla's fastest superchargers max out at 250kW, or a quarter of BYD's claimed feat.

BYD, whose cars are not available in the US, said it will start selling EVs with the Super e-Platform next month — and plans to add 4,000 high-power charging stations in China.

The Chinese juggernaut aims to build more vehicles around the globe with plants in Europe and South America.

Tesla's troubles thus far in 2025 — which has seen the company's stock mired in a nine-week losing streak — are tied to those competitive issues in China as well as depressed sales in the US and Europe. While the new Model Y changeover likely affected sales, the controversial politics of CEO Elon Musk has seen the Tesla brand take a hit with protests at Tesla showrooms and Tesla EV owners turning their back on the brand.

The company also faced a setback in China today, with reports that Tesla is suspending its free trial of its full self-driving (FSD) software on the mainland, pending regulatory approval.
"""

In [11]:
crew = Crew(
    agents=[feature_extractor],
    tasks=[feature_extract_task],
    verbose=True,
)

response = crew.kickoff(inputs={"article": article})

╭─────────────────────────── Crew Execution Started ───────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name: crew                                                                  │
│  ID: e7ecd5c3-3abe-4bb5-b4ac-c0b54491475e                                    │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: 34691960-fe42-4f4c-88c2-66c2be9435d4
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: 34691960-fe42-4f4c-88c2-66c2be9435d4
       Status: Executing Task...
    └── 🤖 Agent: Expert News Article Feature Extractor
            Status: In Progress

# Agent: Expert News Article Feature Extractor
## Task: Extract key feat

In [ ]:
Markdown(response.raw)

The extracted key features from the news articles are as follows:

- **Relevant Company(ies):** Tesla
- **Sentiment Score:** Positive (average rating of 4.5/5, indicating a generally positive sentiment towards Tesla's performance)
- **Publication Date:** December 6, 2023
- **Article Length:** 530 words
- **Keywords:** "Tesla", "BYD", "China rival", "sales surged", "electric vehicles", "EV leader"
- **Sentiment Score Breakdown:**
  - Positive sentiment towards Tesla's performance (e.g., sales surge in China, competitive issues)
  - Neutral or mixed sentiment due to controversies like CEO Elon Musk's politics
  - Negative sentiment towards BYD's success in 2024, with concerns over regulatory approvals and market competition

In [13]:
# for training purposes
f = open("ouptut.md", "w")
f.write(response.raw)
f.close()

In [ ]:
historian_crew = Crew(
    agents=[historian],
    tasks=[historical_context_task],
    verbose=True,
)

result = historian_crew.kickoff(inputs={"file_path": "/home/luukvanroessel/Documents/ML/stock_team/ouptut.md"})


In [ ]:
company_expert_crew = Crew(
    agents=[company_expert],
    tasks=[company_fundamentals_task],
    verbose=True,
)

result = company_expert_crew.kickoff(inputs={"file_path": "/home/luukvanroessel/Documents/ML/stock_team/ouptut.md"})

In [ ]:
result.raw

'the final answer to the original input question'